# Swarm pipeline — Colab setup & playground

This notebook brings up the stigmergic swarm on Colab and exposes the entire knob set as playground cells.

**Storage split**
- **Google Drive (persistent):** run outputs, knowledge base, retrieval cache, Cohere FAISS index. Survives between sessions.
- **Ephemeral `/content/` (per-session):** cloned repo, HuggingFace cache, pip installs. Re-fetched every session.

**How updates work**
- The notebook you're reading right now is a **copy** in your Colab session. It does **not** auto-update from GitHub.
- To get a newer notebook: `File → Open notebook → GitHub` → enter `sfuqua6/Stigmeric-Coordination` → pick `notebooks/colab_setup.ipynb`.
- The **code** in `/content/swarm_repo/` is a `git clone` and *does* update — re-run cell 3 to `git pull`.

**Recommended first-session flow**
1. Cells 1 → 5 in order (mount, paths, clone, install).
2. Cell 6 Cohere setup (API key + HF login + diagnostic).
3. Cell 8 smoke test (`MOCK_LLM=1`) — proves plumbing in ~30 s.
4. Cell 10 real run, or any of the playground cells below.

Every shell cell starts with `cd /content/swarm_repo && ` so cells are independent — you can jump around freely after cell 3 has cloned the repo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Outputs / KB / retrieval cache / Cohere FAISS → Drive. HuggingFace cache → ephemeral `/content/`.

In [ ]:
import os

# Persistent (Drive)
DRIVE_BASE = '/content/drive/MyDrive/swarm'
os.environ['SWARM_OUTPUTS_BASE_DIR']    = f'{DRIVE_BASE}/runs'
os.environ['SWARM_KB_DIR']              = f'{DRIVE_BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR'] = f'{DRIVE_BASE}/retrieval_cache'
os.environ['SWARM_CORPORA_DIR']         = f'{DRIVE_BASE}/corpora'   # Cohere FAISS index

for d in ('runs', 'knowledge_base', 'retrieval_cache', 'corpora'):
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)

# Ephemeral (per-session, /content/)
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

# hf-xet has memory issues on some platforms; the standard downloader is fine.
os.environ['HF_HUB_DISABLE_XET'] = '1'

# Force the Colab tier-aware code paths so config.py sets _TIER from the
# GPU and selects vLLM as the backend. Without this you'll get laptop
# defaults (DeepSeek + GGUF) which won't fit on Colab.
os.environ['COLAB'] = '1'

print('Persistent (Drive):')
for k in ('SWARM_OUTPUTS_BASE_DIR', 'SWARM_KB_DIR', 'SWARM_RETRIEVAL_CACHE_DIR', 'SWARM_CORPORA_DIR'):
    print(f'  {k} = {os.environ[k]}')
print(f"HF_HOME = {os.environ['HF_HOME']}")
print(f"COLAB   = {os.environ['COLAB']}")

## 3. Clone (or pull) the repository

`REPO_URL` is preset to the public Stigmeric-Coordination repo. Re-running this cell on a later session does `git pull` instead of a fresh clone — useful for picking up new commits without restarting Colab.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/sfuqua6/Stigmeric-Coordination.git'
REPO_DIR = '/content/swarm_repo'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'{REPO_DIR} exists; running git pull')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
print('contents:', sorted(os.listdir('.'))[:20])

## 4. Install dependencies

`vllm` is the heavy one (~5 min, brings its own CUDA-matched torch). `cohere`, `datasets`, `faiss-cpu` are the primary retrieval stack. `wikipedia`, `duckduckgo-search`, `requests`, `beautifulsoup4` are the fallback chain.

In [ ]:
!pip install -q -r requirements-colab.txt

# Verify the heavy + retrieval deps imported correctly
!python -c "import vllm; print(f'vllm {vllm.__version__} OK')"
!python -c "import cohere, datasets, faiss; print(f'cohere {cohere.__version__}, datasets {datasets.__version__}, faiss {faiss.__version__} OK')"
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv

## 5. Cohere setup

The primary retrieval path is `core/corpus_store_cohere.py` — pre-computed Cohere embeddings of Simple Wikipedia (~250k articles), indexed in FAISS. The first call downloads ~1 GB to `$SWARM_CORPORA_DIR` (Drive); subsequent sessions reload in <10s.

**Two credentials are needed, for different reasons:**

| What | Why | How to set |
|---|---|---|
| `COHERE_API_KEY` | Embedding YOUR query at runtime (one call per pipeline run) | `os.environ['COHERE_API_KEY']` below |
| HuggingFace token | Downloading the ~1 GB Cohere/Wikipedia dataset from HF Hub | `huggingface-cli login` or `HF_TOKEN` env var |

The dataset is hosted on HuggingFace, not on Cohere's servers — that's why HF auth matters even though the dataset is named "Cohere/…". A common failure mode is setting only the Cohere key and seeing `DatasetNotFoundError`. The cell below sets both and runs a diagnostic to tell you exactly which credential is missing.

In [ ]:
import os

# === EDIT THESE ===
COHERE_API_KEY = ''   # Free key at https://cohere.com/ (~100 queries/min on free tier)
HF_TOKEN       = ''   # Free token at https://huggingface.co/settings/tokens (read-only is fine)
# ==================

if COHERE_API_KEY:
    os.environ['COHERE_API_KEY'] = COHERE_API_KEY
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    # huggingface_hub picks up HF_TOKEN automatically, but newer datasets
    # versions sometimes want HUGGING_FACE_HUB_TOKEN. Set both.
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print('COHERE_API_KEY set:', bool(os.environ.get('COHERE_API_KEY')))
print('HF_TOKEN set:      ', bool(os.environ.get('HF_TOKEN')))

### 5b. Cohere diagnostic

Probes each layer (imports → API key → dataset access → query embedding) and tells you exactly which one fails. Run this before your first real pipeline invocation to catch credential issues early.

In [ ]:
# Step-by-step Cohere readiness check
import os

print('=== Step 1: imports ===')
try:
    import cohere, datasets, faiss
    print(f'  cohere   {cohere.__version__}')
    print(f'  datasets {datasets.__version__}')
    print(f'  faiss    {faiss.__version__}')
except ImportError as e:
    print(f'  FAIL: {e}')
    print('  Fix: !pip install -q -r requirements-colab.txt')
    raise SystemExit

print('\n=== Step 2: Cohere API key ===')
key = os.environ.get('COHERE_API_KEY', '')
if not key:
    print('  FAIL: COHERE_API_KEY not set. Set it in cell 5 above and re-run.')
    raise SystemExit
try:
    co = cohere.Client(key)
    resp = co.embed(texts=['health check'], model='embed-multilingual-v2.0', input_type='search_query')
    print(f'  OK ({len(resp.embeddings[0])}-dim embedding returned)')
except Exception as e:
    print(f'  FAIL: {type(e).__name__}: {e}')
    print('  Fix: re-check key at https://dashboard.cohere.com/api-keys')
    raise SystemExit

print('\n=== Step 3: HuggingFace dataset access (Cohere/wikipedia-22-12-simple-embeddings) ===')
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN') or None)
    info = api.dataset_info('Cohere/wikipedia-22-12-simple-embeddings')
    print(f'  OK (dataset accessible, gated={getattr(info, "gated", "unknown")})')
except Exception as e:
    print(f'  FAIL: {type(e).__name__}: {e}')
    print('  Fix: set HF_TOKEN in cell 5 above. The dataset is on HF Hub and may be gated.')
    print('       You can also do !huggingface-cli login in a separate cell.')
    raise SystemExit

print('\n=== Step 4: full FAISS path (cached or first-time download) ===')
print('  This will download ~1 GB on first run; <10s on subsequent runs.')
import sys
sys.path.insert(0, '/content/swarm_repo')
try:
    from core.corpus_store_cohere import get_store
    store = get_store()
    chunks = store.search('renewable energy', n_chunks=5)
    print(f'  OK: retrieved {len(chunks)} chunks for sample query')
    for c in chunks[:2]:
        print(f'    [{c.source_tag}] {c.text[:80]}...')
except Exception as e:
    print(f'  FAIL: {type(e).__name__}: {e}')
    raise SystemExit

print('\nAll Cohere layers ready. Run pipeline with real retrieval next.')

## 6. Smoke test (MockLLM, no model load)

Verifies the pipeline plumbing in ~30 s. MockLLM emits SHA1-seeded phrases — what you're checking is that the run completes end-to-end and writes a directory to Drive, not that the content makes sense.

In [ ]:
!cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py debate "Test thesis" --corpus=placeholder

## 7. Real run

Loads Qwen2.5-7B-Instruct via vLLM. On T4 (16 GB VRAM), the loader walks a cascade in `core/llm.py:_build_cascade` — configured AWQ first, then fp16, then progressively smaller models. Watch the `[llm-cascade]` log: the first `SUCCESS at attempt N/M` line tells you what loaded.

Cohere retrieval runs as part of this — you should see `[retrieval] source=cohere_wiki_simple, N=20` early in the log. Expect ~90–120 min for a 3-round debate on T4.

In [ ]:
!cd /content/swarm_repo && python run_swarm.py debate "Does free will exist?" 

## 8. Inspect outputs

Every run drops `answer.txt`, `signals.json`, `summary.json`, `round_log.json`, `citations.json`, `lineage.dot`, `run_meta.json` into a timestamped subdir under `$SWARM_OUTPUTS_BASE_DIR/outputs/`.

In [ ]:
import json
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if runs:
    latest = runs[-1]
    print('latest run:', latest)
    print()
    summary_path = latest / 'summary.json'
    if summary_path.exists():
        print('=== summary.json ===')
        print(json.dumps(json.loads(summary_path.read_text()), indent=2))
        print()
    answer_path = latest / 'answer.txt'
    if answer_path.exists():
        print('=== answer.txt ===')
        print(answer_path.read_text())
else:
    print('no real runs yet — run cell 7, or check outputs_mock/ for cell 6 results')

---

# Playground

Below are preset experiments. Each cell is independent — once cells 1-5 have run you can hop around freely. **Copy a cell (Ctrl+M, A) to fork an experiment without losing the preset.**

The pattern throughout: env vars are set BEFORE the `!python` invocation. `core/config.py` reads `SWARM_*` env vars on import and bakes them into module-level constants. Restart the kernel + re-run cells 1-5 only if you need to change something that's not exposed as a `SWARM_*` knob.

## P1. Try different task types

`run_swarm.py` supports five task types. Each gates a different set of roles (`ROLES_FOR_TASK` in `run_swarm.py`):

- `debate` — full pipeline incl. Hater + Validator
- `analysis` — full pipeline incl. Hater + Validator
- `problem_solving` — Validator suppressed
- `creative` — Validator + Hater suppressed
- `coding` — swaps in `agents/coding_roles.py`

Use `MOCK_LLM=1` for a quick plumbing tour without burning GPU time.

In [ ]:
# Mock-mode tour (uncomment one at a time)
!cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py problem_solving "How can cities reduce traffic?" --corpus=placeholder
# !cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py creative         "Write a haiku about emergence" --corpus=placeholder
# !cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py analysis         "What causes innovation?" --corpus=placeholder
# !cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py coding           "Implement a binary search" --corpus=placeholder

## P2. Pipeline flags

| Flag | Effect |
|---|---|
| `--mode=baseline` | Disable signal store / partitioning / provenance boost. A/B comparison condition for the stigmergic hypothesis. |
| `--corpus=placeholder` | Skip web retrieval, use engineered corpus. Diversity numbers from this mode are not empirical evidence. |
| `--ignore-kb` | Don't consult the cross-run knowledge base. |
| `--reset-kb` | Quarantine existing KB entries before this run. |
| `--show-partition-overlap` | Surface Jaccard input-overlap diagnostics in the round log. |
| `--cloud-validator=anthropic` / `=gemini` | Cloud-LLM validator (needs `ANTHROPIC_API_KEY` / `GEMINI_API_KEY`). |

The pair below produces stigmergic-vs-baseline runs on the same prompt — diff them with the comparison cell further down.

In [ ]:
# Stigmergic (default)
!cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py debate "Should AI development be paused?" \
    --corpus=placeholder --show-partition-overlap

In [ ]:
# Baseline (no signal store, no partitioning)
!cd /content/swarm_repo && MOCK_LLM=1 python run_swarm.py debate "Should AI development be paused?" \
    --mode=baseline --corpus=placeholder

## P3. Rounds and iterations

These are the highest-impact knobs for the quality/wall-time tradeoff. Defaults on T4 are 3 rounds × 10 iterations per round (90 LLM calls / round before factoring in agent counts).

| Env var | Default (T4) | What it does |
|---|---|---|
| `SWARM_NUM_ROUNDS` | 3 | How many rounds of (Phase A + Phase B + decay/prune) |
| `SWARM_ITERATIONS_PER_ROUND` | 10 | How many times each agent runs per round |
| `SWARM_SCOUT_MAX_DEPOSITS` | 2 | Scout saturation cap (stop after N successful deposits) |

Rule of thumb: more rounds = better consensus stability; more iterations = denser per-round output. Doubling either roughly doubles wall time.

In [ ]:
# Longer run: 5 rounds * 15 iterations
!cd /content/swarm_repo && \
    SWARM_NUM_ROUNDS=5 SWARM_ITERATIONS_PER_ROUND=15 \
    python run_swarm.py debate "What makes democracies stable?" 

In [ ]:
# Quick run: 2 rounds * 6 iterations (good for iterating on prompts)
!cd /content/swarm_repo && \
    SWARM_NUM_ROUNDS=2 SWARM_ITERATIONS_PER_ROUND=6 \
    python run_swarm.py debate "What makes democracies stable?" 

## P4. Agent population

| Env var | Default (T4) | Role |
|---|---|---|
| `SWARM_NUM_SCOUTS` | 6 | Read disjoint corpus partitions, produce initial claims |
| `SWARM_NUM_FORAGERS` | 6 | Develop claims; sampling strategies differ per agent |
| `SWARM_NUM_CRITICS` | 3 | Score / dispute / corroborate claims |
| `SWARM_NUM_HATERS` | 2 | Adversarial challenges to consensus clusters |
| `SWARM_NUM_VALIDATORS` | 2 | External fact-check (Wikipedia / cloud LLM) |

Bigger populations = more diversity per round but more wall time. The Jaccard partition-overlap report (in `--show-partition-overlap`) tells you if scouts are getting genuinely disjoint inputs.

In [ ]:
# Cranked-up populations: 8 scouts, 8 foragers, 4 critics, 3 haters
!cd /content/swarm_repo && \
    SWARM_NUM_SCOUTS=8 SWARM_NUM_FORAGERS=8 SWARM_NUM_CRITICS=4 SWARM_NUM_HATERS=3 \
    python run_swarm.py debate "Is universal basic income economically viable?" 

In [ ]:
# Lean run: minimal pop for fast iteration
!cd /content/swarm_repo && \
    SWARM_NUM_SCOUTS=3 SWARM_NUM_FORAGERS=3 SWARM_NUM_CRITICS=2 SWARM_NUM_HATERS=1 SWARM_NUM_VALIDATORS=1 \
    MOCK_LLM=1 python run_swarm.py debate "Test thesis" --corpus=placeholder

## P5. Model + token budgets

| Env var | Default (T4) | What it does |
|---|---|---|
| `SWARM_MODEL` | `Qwen/Qwen2.5-7B-Instruct` | Base model |
| `SWARM_BACKEND` | auto (`vllm` on Colab) | Force a specific backend |
| `SWARM_AWQ_MODEL` | auto (`<base>-AWQ`) | Override the AWQ variant used in cascade stage 2 |
| `VLLM_DTYPE` | `float16` (T4/L4) / `bfloat16` (A100) | vLLM weight dtype |
| `SWARM_GPU_MEM` | tier-aware (13/20/36/76 GiB) | HF backend GPU budget (vLLM ignores) |
| `SWARM_PROMPT_MAX_LEN` | 1024 | Tokenizer truncation cap |
| `SWARM_MAX_TOKENS_SCOUT` | 140 | Per-call token budget for scouts |
| `SWARM_MAX_TOKENS_FORAGER` | 200 | Foragers/developers |
| `SWARM_MAX_TOKENS_CRITIC` | 150 | Critics |
| `SWARM_MAX_TOKENS_HATER` | 200 | Haters |
| `SWARM_MAX_TOKENS_VALIDATOR` | 100 | Validators |
| `SWARM_MAX_TOKENS_SYNTHESIZER` | 800 | Synthesizer (final answer per cluster) |

In [ ]:
# Switch to smaller 3B model for faster iteration
!cd /content/swarm_repo && \
    SWARM_MODEL='Qwen/Qwen2.5-3B-Instruct' \
    python run_swarm.py debate "Is consciousness computable?" 

In [ ]:
# Boost synthesizer token budget for longer per-cluster answers
!cd /content/swarm_repo && \
    SWARM_MAX_TOKENS_SYNTHESIZER=1500 \
    python run_swarm.py analysis "What drives technological adoption?" 

## P6. Survival filter thresholds

These control how strict the projection is when deciding which clusters survive into the final answer. Read `core/projection.py:_apply_survival_filter` for the full logic; the four buckets are `surviving`, `contested`, `weakly_supported`, `rejected_by_field`, plus a fifth `unverified` for clusters that structurally pass but don't clear the credibility gate.

| Env var | Default | What it does |
|---|---|---|
| `SWARM_SURVIVAL_MIN_SUPPORT_DIVERSITY` | 3 | Min distinct forager strategies → not `weakly_supported` |
| `SWARM_SURVIVAL_REJECT_DISSENT_PRESSURE` | 1.5 | Above this → `rejected_by_field` |
| `SWARM_SURVIVAL_CONTEST_MIN` | 0.5 | Lower bound of `contested` range |
| `SWARM_SURVIVAL_CONTEST_MAX` | 1.5 | Upper bound of `contested` range |
| `SWARM_SURVIVAL_VERIFY_MIN` | 0.3 | Verification score that clears the credibility gate |
| `SWARM_SURVIVAL_BROAD_SUPPORT` | 4 | Support diversity that alone clears credibility gate |
| `SWARM_CLUSTER_SIM_THRESHOLD` | 0.72 (Colab) | Cosine sim above which INITIALs merge into one cluster |

If you're getting too many surviving clusters: raise `SWARM_SURVIVAL_MIN_SUPPORT_DIVERSITY` to 4 or `SWARM_SURVIVAL_BROAD_SUPPORT` to 5. If you're getting too few: lower them to 2.

In [ ]:
# Stricter: require 4+ supporters and broader credibility evidence
!cd /content/swarm_repo && \
    SWARM_SURVIVAL_MIN_SUPPORT_DIVERSITY=4 \
    SWARM_SURVIVAL_BROAD_SUPPORT=5 \
    python run_swarm.py debate "Is nuclear power a viable climate solution?" 

In [ ]:
# Looser: accept clusters with only 2 supporters (use cautiously — bloats outputs)
!cd /content/swarm_repo && \
    SWARM_SURVIVAL_MIN_SUPPORT_DIVERSITY=2 \
    SWARM_SURVIVAL_BROAD_SUPPORT=3 \
    python run_swarm.py debate "Is nuclear power a viable climate solution?" 

In [ ]:
# Tighter clustering: more clusters, each more specific
!cd /content/swarm_repo && \
    SWARM_CLUSTER_SIM_THRESHOLD=0.85 \
    python run_swarm.py analysis "What patterns emerge in successful tech companies?" 

## P7. Diagnose

`diagnose.py` runs a self-check on signal store + pipeline plumbing. Useful when a real run produces empty outputs or NaN strengths — isolates the layer.

In [ ]:
!cd /content/swarm_repo && python diagnose.py

## P8. Compare two runs

`tools/compare_runs.py` diffs two `summary.json` files. Common use: stigmergic vs baseline twin from cell P2.

In [ ]:
import os
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []

if len(runs) >= 2:
    a, b = runs[-2], runs[-1]
    print(f'comparing {a.name} vs {b.name}')
    !cd /content/swarm_repo && python tools/compare_runs.py "{a}" "{b}"
else:
    print(f'need at least 2 runs in {outputs_root}; have {len(runs)}')

## P9. Re-synthesize from a saved store

`synthesize.py` re-renders the final answer from a saved signal store. Useful for swapping the synthesizer model without re-running the whole pipeline.

In [ ]:
# Re-synthesize the most recent run
import os
from pathlib import Path

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []
if runs:
    RUN_PATH = runs[-1]
    print(f'synthesizing from {RUN_PATH}')
    !cd /content/swarm_repo && python synthesize.py "{RUN_PATH}"
else:
    print('no runs yet')

## P10. Visualize the signal DAG

Every real run drops `lineage.dot` — Graphviz DAG of signal ancestry. Render inline.

In [ ]:
import os, subprocess
from pathlib import Path
from IPython.display import Image, display

outputs_root = Path(os.environ['SWARM_OUTPUTS_BASE_DIR']) / 'outputs'
runs = sorted(outputs_root.glob('*'), key=lambda p: p.stat().st_mtime) if outputs_root.exists() else []
if runs and (runs[-1] / 'lineage.dot').exists():
    dot_path = runs[-1] / 'lineage.dot'
    png_path = runs[-1] / 'lineage.png'
    # graphviz preinstalled on Colab; if not: !apt-get install -y graphviz
    subprocess.run(['dot', '-Tpng', str(dot_path), '-o', str(png_path)], check=True)
    display(Image(str(png_path)))
else:
    print('no runs with lineage.dot found')

## P11. Phase-isolated orchestrator (crash-resumable)

Spawns short-lived subprocesses, one per phase per round + synth. Slower (subprocess startup) but **crash-resumable** — if a phase fails, re-run with the same `--run-id` and it skips completed phases. Re-running across Colab disconnects works as long as the run dir is on Drive.

In [ ]:
!cd /content/swarm_repo && python tools/run_isolated.py debate "Does free will exist?" --run-id=free_will_isolated

---

## Troubleshooting

- **Cohere `DatasetNotFoundError`.** The dataset is on HuggingFace Hub. Set `HF_TOKEN` in cell 5 (not just `COHERE_API_KEY`). Re-run cell 5b diagnostic — step 3 will report this specifically.
- **Cohere `401 Unauthorized`.** Cohere key is invalid / expired. Re-check at https://dashboard.cohere.com/api-keys.
- **`/content/swarm_repo/run_swarm.py` doesn't exist.** Cell 3 failed — check its output for `git clone` errors. Most common: typo in `REPO_URL`.
- **`[llm] all HF attempts failed; falling back to MockLLM`.** Look at the `[llm-cascade]` log: which attempt failed and why (OOM, network, version)? Pipeline still completes but the answer is gibberish. Don't trust outputs after this banner.
- **Colab disconnects mid-run.** Use the phase-isolated orchestrator (P11) with a pinned `--run-id`. Checkpoints on Drive. Re-run the same command to resume.
- **`vllm` install fails.** CUDA wheel URLs change. Try `!pip install --upgrade pip` first.
- **HF `memory allocation` error.** That's `hf-xet`. Cell 2 sets `HF_HUB_DISABLE_XET=1`. If you still see it: `!pip uninstall -y hf-xet` and re-run.
- **VRAM OOM.** Set `SWARM_MODEL='Qwen/Qwen2.5-3B-Instruct'` and/or `SWARM_PROMPT_MAX_LEN=512`.
- **Notebook out of date.** It's a copy that doesn't auto-sync. `File → Open notebook → GitHub` to re-open fresh. The cloned *code* under `/content/swarm_repo/` does pull via cell 3.
- **Idle disconnect.** Free Colab disconnects after ~90 min idle. Keep tab focused, or in DevTools console:
  ```javascript
  setInterval(() => document.querySelector('colab-toolbar-button#connect')?.click(), 60000);
  ```